# Bangladesh Multi-Tool AI Agent — Colab Demo
Execute top-to-bottom. This notebook runs from the project repository, which contains the `src/` modules. Public dataset download and hosted LLM/Tavily calls require internet access.

## 0 — Upload and unzip the project
Upload the downloaded `bangladesh-multitool-agent.zip`. This is only needed when you are not running from an already cloned repository.

In [ ]:
from google.colab import files
import zipfile, os
if not os.path.exists("/content/bangladesh-multitool-agent/src"):
    uploaded = files.upload()
    zip_name = next((name for name in uploaded if name.endswith(".zip")), None)
    if not zip_name:
        raise FileNotFoundError("Upload bangladesh-multitool-agent.zip")
    with zipfile.ZipFile(zip_name) as archive:
        archive.extractall("/content")
%cd /content/bangladesh-multitool-agent

## 1 — Environment and dependencies
**Purpose:** install the AgentExecutor stack. **Why:** Colab does not automatically provide every library. **Behind the scenes:** pip resolves and installs Python wheels. **Alternative:** modern LangChain `create_agent` (but this assignment requires `AgentExecutor`).

In [ ]:
%pip -q install -r requirements.txt
%pip check

## 2 — Download CSV datasets and create SQLite files
**Purpose:** build three persistent-on-disk tables with explicit types. **Why:** agent tools query structured data, not raw CSV. **Behind the scenes:** Hugging Face file discovery → Pandas normalize → SQLite `CREATE TABLE` and `INSERT`. **Alternative:** SQLite CLI import or SQLAlchemy.

In [ ]:
!python -m src.build_databases --data-dir data

## 3 — Inspect real schemas
**Why:** do not assume a field such as beds, doctor count, or cuisine exists.

In [ ]:
import sqlite3
for table in ["institutions", "hospitals", "restaurants"]:
    with sqlite3.connect(f"data/{table}.db") as db:
        print("\n", table, "records:", db.execute(f'SELECT COUNT(*) FROM "{table}"').fetchone()[0])
        print([(r[1], r[2]) for r in db.execute(f'PRAGMA table_info("{table}")')])

## 4 — Load Colab Secrets
Create `OPENAI_API_KEY` and `TAVILY_API_KEY` under Colab **Secrets**. Enable notebook access. Do not print keys.

In [ ]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")
print("API keys loaded (values hidden).")

## 5 — Test each tool directly
**Purpose:** isolate tool connectivity before LLM orchestration. **Behind the scenes:** read-only SQLite `SELECT` and a Tavily request. **Alternative:** substitute SerpAPI/Bing in the `WebSearchTool` function.

In [ ]:
from src.tools import ALL_TOOLS, institutions_db_tool, hospitals_db_tool, restaurants_db_tool, web_search_tool
print(hospitals_db_tool.invoke({"sql_query": "SELECT COUNT(*) AS total FROM hospitals WHERE LOWER(district)='dhaka'"}))
print(institutions_db_tool.invoke({"sql_query": "SELECT name, district FROM institutions LIMIT 2"}))
print(restaurants_db_tool.invoke({"sql_query": "SELECT name, rating FROM restaurants LIMIT 2"}))
print(web_search_tool.invoke({"query": "official DGHS Bangladesh role website"})[:1000])

## 6 — Create LangChain AgentExecutor
**Purpose:** let the LLM select one or more tools. **Behind the scenes:** LangChain sends tool schemas to the model; AgentExecutor runs tool calls and returns observations to the model.

In [ ]:
from src.agent import create_executor
agent_executor = create_executor()
print("Registered:", [t.name for t in ALL_TOOLS])

## 7 — Evaluate sample queries
**Purpose:** show final answers and tool-selection traces. **Alternative:** evaluate factual accuracy against known SQL counts.

In [ ]:
questions = [
    "How many healthcare facilities are in Dhaka district?",
    "How many institutions are in Rajshahi district?",
    "Show five top-rated restaurants in Dhaka using recorded ratings.",
    "What is the role of DGHS in Bangladesh?",
]
for question in questions:
    print("\nQUESTION:", question)
    result = agent_executor.invoke({"input": question})
    print("ANSWER:", result["output"])
    print("TOOLS:", [action.tool for action, _ in result.get("intermediate_steps", [])])

## 8 — Test SQL security (no LLM/API usage)

In [ ]:
print(hospitals_db_tool.invoke({"sql_query": "DELETE FROM hospitals"}))
print(hospitals_db_tool.invoke({"sql_query": "SELECT name, beds FROM hospitals LIMIT 2"}))

## 9 — Offline unit tests

In [ ]:
!python -m unittest discover -s tests -v

## Assignment notes
The notebook requires a cloned/uploaded project in the current Colab working directory. Preserve outputs by saving the notebook after running. Dataset and API results depend on available sources and credentials. No live demo link has been published automatically.